# Time Series with Pandas

To deal with dates and times, Pandas provides proper efficient objects in a simple data version, and in an index version:
- **time stamps**: precise time instants, e.g. July 15th 1985 $\implies$<br> 
Pandas objects: ${\tt Timestamps/DatetimeIndex}$
<br>

- **time intervals-periods**: the length of time slot comprised between an initial and a final time instant, e.g. 2024 year. Periods are not overlapping time intervals with uniform length, e.g. 24 hours of a day $\implies$<br> 
Pandas objects: ${\tt Period/PeriodIndex}$
<br>

- **time deltas (durations)**: the exact length of time, e.g. 9.58 seconds $\implies$<br>
Pandas objects: ${\tt Timedelta/TimedeltaIndex}$

Method ${\tt pd.to\_datetime()}$ create a ${\tt Timestamps}$ from a single date and a ${\tt DatetimeIndex}$ from a list of dates, and ${\tt pd.to\_period()}$ convert a ${\tt DatetimeIndex}$ to a ${\tt PeriodIndex}$.<br>
The standard frequencies used in Pandas are reported in the following table ![alt text](freq.png)
Consider the following examples.

In [ ]:
from datetime import datetime # datetime is a collection of utilities for date and time of the 
                              # Python standard library
import pandas as pd
import matplotlib.pyplot as plt


single_date = pd.to_datetime('14th of May, 2000')
multiple_dates = pd.to_datetime([datetime(2015, 7, 3), '4th of July, 2015',\
                       '2015-Jul-6', '07-07-2015', '20150708'])
print(single_date)
print('//////////')
print(multiple_dates)

In [ ]:
# a DatetimeIndex can be converted to a PeriodIndex by specifying the period freqeuncy

print(multiple_dates.to_period('D')) # input 'D' indicates daily frequency of the period

In [ ]:
# a TimedeltaIndex can be created by subtracting a Timestamp from a DatetimeIndex

print(multiple_dates - multiple_dates[0])

${\tt DatetimeIndex}$, ${\tt PeriodIndex}$, and ${\tt TimedeltaIndex}$ can be quickly created in an automated way by using ${\tt date\_range()}$, ${\tt period\_range()}$, and ${\tt timedelta\_range()}$.<br>
Consider the following examples.

In [ ]:
pd.date_range('2015-07-03', '2015-07-10')

In [ ]:
pd.date_range('2015-07-03', periods=8)

In [ ]:
pd.date_range('2015-07-03', periods=8, freq='h')

In [ ]:
pd.period_range('2015-07', periods=8, freq='M')

In [ ]:
pd.timedelta_range(0, periods=10, freq='h')

The following additional methods can be used to handle dates and times:
- ${\tt resample()}$: combined with an aggregation function (${\tt mean()}$, ${\tt std()}$,...) computes an aggregated resampling of data on a specified time frequency
- ${\tt asfreq()}$: select a subset of data according to a specified time frequency
- ${\tt shift()}$: translates data by a specifid time offset 

# A)
Import and plot the daily close price of the Google stocks from 2004 to 2016.

In [ ]:
goog = pd.read_csv('GOOG.csv', delimiter=',')
goog

In [ ]:
goog['Date'] = pd.to_datetime(goog['Date'])
goog.set_index('Date', inplace = True)
print(goog.index)
goog = goog['Close']
print(goog)

In [ ]:
goog.plot()

# B)

Resample the Google stocks price with different frequencies (down-sample and up-sample) and with and without aggregation.

In [ ]:
# down-sample

goog.plot(alpha=0.5, style='-')
goog.resample('BYE').mean().plot(style=':') 
goog.asfreq('BYE').plot(style='--'); 
plt.legend(['input', 'resample', 'asfreq'],loc='upper left');

In [ ]:
partial_data = goog.iloc[:10]
partial_data

In [ ]:
# up-sample

fig, ax = plt.subplots(2, sharex=True) 
partial_data.asfreq('D').plot(ax=ax[0], marker='o')
partial_data.asfreq('D', method='bfill').plot(ax=ax[1], style='-o')  
partial_data.asfreq('D', method='ffill').plot(ax=ax[1], style='--o')
ax[1].legend(["back-fill", "forward-fill"]);

# C)
Exploit the ${\tt shift()}$ method to operate the time series differencing, plot it, and check stationarity.

In [ ]:
goog_diff = goog.shift(-1)-goog

fig, ax = plt.subplots(2, sharey=True)
goog.plot(ax=ax[0])
goog_diff.plot(ax=ax[1])

# D)
Exploit again the ${\tt shift()}$ method to generate the time series of the Google stocks one-year Return On Investment (ROI), computed as $\mbox{ROI}=100*(\frac{price_{t+365}-price_t}{price_t})$, in order determine which was the best investment period.

In [ ]:
ROI = 100 * ((goog.shift(-365)-goog)/goog) # a negative shift anticipate data
ROI.plot()

# E)

Observe the difference in terms of 10 lags autocorrelation plot of the of the (business) quarterly and yearly Google stocks price and differenced Google stocks price. 

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(goog.resample('BQE').mean(), lags=10, missing='drop', title='Google quarterly mean')
plot_acf(goog.resample('BYE').mean(), lags=10, missing='drop', title='Google yearly mean')

plot_acf(goog_diff.resample('BQE').mean(), lags=10, missing='drop', title='Google quarterly mean (diff.)')
plot_acf(goog_diff.resample('BYE').mean(), lags=10, missing='drop', title='Google yearly mean (diff.)')

plt.show()